# 📍 05 · Keypoints and Particles
### A complete point-detection pipeline, from low-pass filtering to sub-pixel localization

**Part C · Finding structure** · Module 05 of 29 · *BiologyToBioimage*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/sci_img/tutorial/BiologyToBioimage/notebooks/05_keypoints_and_particles.ipynb)

| | |
|---|---|
| **Hardware target** | CPU only (no GPU needed) |
| **Core frameworks** | NumPy, SciPy |
| **Prerequisites** | Modules 02–04 |
| **Estimated time** | 90–105 minutes |
| **02-740 topics covered** | Point/particle feature detection pipeline: low-pass filtering, local maxima/background, Delaunay-triangulation correspondence, statistical (t-test) feature selection, sub-pixel Gaussian-fitting localization |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> The particle images below are SIMULATED with known ground-truth positions; nothing is downloaded.

## 🎯 Learning objectives

1. Define what a "particle" or "point feature" means in bioimage analysis, and explain why so many
   cellular structures appear this way.
2. Implement the full classical particle-detection pipeline: low-pass filtering, local-maximum and
   local-background detection, and statistical feature selection.
3. Use Delaunay triangulation to establish correspondence between detected maxima and minima.
4. Implement sub-pixel localization by Gaussian fitting, and explain why pixel-level precision is
   often not good enough.

## 🗺️ Contents

1. What is a particle, and why does point detection matter in bioimaging?
2. Step 1: low-pass filtering for noise suppression
3. Steps 2–3: local maxima, local background, and correspondence via Delaunay triangulation
4. Step 4: statistical feature selection with a t-test
5. Sub-pixel localization by Gaussian fitting

## 🧫 The biological question

Axonal cargo transport — the directed movement of mitochondria, proteins, and other cargo along
microtubules inside a neuron's long, thin axon — appears in a fluorescence microscope as small,
moving, bright dots against a noisy background. The same is true for fluorescent speckles marking
individual protein complexes in a mitotic spindle. In both cases, the underlying biological objects
are small enough, relative to the microscope's diffraction limit, that they appear not as resolved
shapes but as **point features** — "particles" in bioimaging terminology (the field also uses "point
detection" and "single-particle detection" essentially interchangeably). Getting accurate,
sub-pixel-precise positions out of these noisy blobs is the foundation for everything downstream: you
cannot measure a transport *speed* (Module 14, object tracking) without first getting a precise
transport *position* at every time point.

---
## Environment setup

In [ ]:
# ── Hardware check ──
import platform
print('Python', platform.python_version())
try:
    import torch
    print('Torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
except ImportError:
    print('PyTorch not installed yet — the next cell installs it if needed.')


In [ ]:
!pip install -q scikit-image
# ── Dependencies ── (Colab already has most of these; we install only what is missing)


In [ ]:
# ── Global imports and configuration ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.spatial import Delaunay
from scipy import stats
SEED = 5
rng = np.random.default_rng(SEED)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.titlesize'] = 11


### 🧰 Course toolkit

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v1.0; copied so this notebook runs alone) ──
import numpy as np
import matplotlib.pyplot as plt
from scipy import ndimage as ndi
from scipy.optimize import linear_sum_assignment

def gaussian_psf(sigma: float, radius: int | None = None) -> np.ndarray:
    """A normalised 2-D Gaussian point-spread function (the microscope's blur kernel)."""
    if radius is None:
        radius = max(1, int(np.ceil(3 * sigma)))
    ax = np.arange(-radius, radius + 1)
    xx, yy = np.meshgrid(ax, ax)
    k = np.exp(-(xx ** 2 + yy ** 2) / (2 * sigma ** 2))
    return k / k.sum()


def blur_image(image: np.ndarray, psf: np.ndarray) -> np.ndarray:
    """Convolve an image with a point-spread function (microscope optical blur)."""
    return ndi.convolve(image, psf, mode="reflect")


def add_gaussian_noise(image: np.ndarray, sigma: float, rng: np.random.Generator) -> np.ndarray:
    """Additive Gaussian (read) noise: the classic noise floor of a camera sensor."""
    return image + rng.normal(0.0, sigma, size=image.shape)


def _draw_blob(canvas: np.ndarray, cy: float, cx: float, r: float, shape: str,
                rng: np.random.Generator, angle: float = 0.0) -> None:
    """Add one synthetic nucleus/cell of the given shape class onto ``canvas`` (in place, additive)."""
    size = canvas.shape[0]
    yy, xx = np.meshgrid(np.arange(size), np.arange(size), indexing="ij")
    y, x = yy - cy, xx - cx
    ca, sa = np.cos(angle), np.sin(angle)
    yr, xr = ca * y + sa * x, -sa * y + ca * x  # rotate into the blob's own frame
    if shape == "round":
        d = (yr / r) ** 2 + (xr / r) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "elongated":
        d = (yr / (r * 0.55)) ** 2 + (xr / (r * 1.7)) ** 2
        canvas += np.exp(-d * 2.0)
    elif shape == "kidney":
        d = (yr / r) ** 2 + (xr / r) ** 2
        dent = np.exp(-((yr - 0.0) ** 2 + (xr - r * 0.9) ** 2) / (2 * (r * 0.5) ** 2))
        canvas += np.clip(np.exp(-d * 2.0) - 0.9 * dent, 0, None)
    elif shape == "ring":
        d = np.sqrt(yr ** 2 + xr ** 2)
        canvas += np.exp(-((d - r) ** 2) / (2 * (r * 0.28) ** 2))
    elif shape == "dividing":
        sep = r * 1.1
        for sign in (-1, 1):
            d = ((yr - sign * sep * 0.5) / (r * 0.75)) ** 2 + (xr / (r * 0.75)) ** 2
            canvas += np.exp(-d * 2.0)
    else:  # irregular: a round blob perturbed by a few random bumps
        d = (yr / r) ** 2 + (xr / r) ** 2
        base = np.exp(-d * 2.0)
        for _ in range(3):
            bx, by = rng.uniform(-r, r, size=2)
            bump = np.exp(-(((yr - by) ** 2 + (xr - bx) ** 2) / (2 * (r * 0.45) ** 2)))
            base = np.clip(base + 0.5 * bump, 0, 1.3)
        canvas += base


def simulate_two_channel_image(size: int = 160, n_spots: int = 60, association: float = 0.7,
                                psf_sigma: float = 1.3, seed: int = 0):
    """Two SIMULATED fluorescence channels (protein A, protein B) with a known association.

    ``association`` in [0, 1] is the fraction of channel-B spots placed near an existing
    channel-A spot (true colocalization); the rest are placed independently at random. Returns
    ``chA``, ``chB`` (size x size images), ``points_a``, ``points_b`` (coordinates) and
    ``true_association`` (the parameter used, for the ground-truth check).
    """
    rng = np.random.default_rng(seed)
    margin = size * 0.08
    pts_a = rng.uniform(margin, size - margin, size=(n_spots, 2))
    pts_b = []
    for i in range(n_spots):
        if rng.random() < association:
            jitter = rng.normal(0, size * 0.015, size=2)
            pts_b.append(np.clip(pts_a[i] + jitter, margin, size - margin))
        else:
            pts_b.append(rng.uniform(margin, size - margin, size=2))
    pts_b = np.array(pts_b)

    def _render(pts):
        canvas = np.zeros((size, size))
        for cy, cx in pts:
            _draw_blob(canvas, cy, cx, size * 0.018, "round", rng)
        canvas = blur_image(canvas, gaussian_psf(psf_sigma))
        return add_gaussian_noise(canvas, 0.02, rng).clip(0, None)

    chA, chB = _render(pts_a), _render(pts_b)
    return dict(chA=chA, chB=chB, points_a=pts_a, points_b=pts_b,
                true_association=association, source="SIMULATED")


def fit_gaussian_spot(patch: np.ndarray):
    """Sub-pixel localisation of one bright spot: fit a 2-D Gaussian's centroid by intensity-
    weighted moments (the simple, closed-form version of 'Gaussian fitting' spot localisation)."""
    yy, xx = np.meshgrid(np.arange(patch.shape[0]), np.arange(patch.shape[1]), indexing="ij")
    w = np.clip(patch - patch.min(), 0, None)
    total = w.sum()
    if total <= 0:
        return patch.shape[0] / 2, patch.shape[1] / 2
    cy = (w * yy).sum() / total
    cx = (w * xx).sum() / total
    return cy, cx


def show_image(ax, image: np.ndarray, title: str = "", cmap: str = "gray"):
    """Show one image on a given Matplotlib axis, titled, with ticks removed."""
    ax.imshow(image, cmap=cmap)
    ax.set_title(title, fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])


def show_image_grid(images, titles=None, ncols: int = 4, cmap: str = "gray", figsize_scale: float = 2.6):
    """A grid of images with Matplotlib, one subplot each, titles optional."""
    n = len(images)
    ncols = min(ncols, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(figsize_scale * ncols, figsize_scale * nrows))
    axes = np.atleast_1d(axes).ravel()
    for i, img in enumerate(images):
        show_image(axes[i], img, titles[i] if titles else "", cmap=cmap)
    for ax in axes[n:]:
        ax.axis("off")
    fig.tight_layout()
    return fig, axes

---
## 1 · What is a particle, and why does point detection matter?

From an image-analysis perspective, one useful working definition: **a particle is a local
intensity maximum whose level is substantially higher than its local background.** Two pieces of
information matter most once a particle is found: its **position** (often needed at sub-pixel
resolution, since real biological objects rarely sit exactly on a pixel center) and its **intensity**
(which, within the diffraction limit, can carry information about how many molecules are present at
that location).

In [ ]:
# ── 1.1 simulate a field of particles, the way axonal-transport vesicles might appear ──
sim = simulate_two_channel_image(size=150, n_spots=35, association=0.0, psf_sigma=1.4, seed=SEED)
particles_image = sim['chA']
true_positions = sim['points_a']   # the GROUND TRUTH (row, col) positions, known only because this is simulated

show_image_grid([particles_image], ['simulated particle field (e.g. transported vesicles)'], ncols=1, figsize_scale=4)
plt.show()
print('number of true particles:', len(true_positions))

🔍 **What just happened?** `simulate_two_channel_image` placed 35 small blobs (we are only using
channel A here) at known positions, blurred them with a point-spread function, and added Gaussian
noise — exactly the degradation chain from Module 02. `true_positions` is information a real
experiment never directly has; we keep it here purely to grade our own detection pipeline, which is
the whole point of building these checks into simulated data.

---
## 2 · Step 1: low-pass filtering for noise suppression

The classical particle-detection pipeline begins by low-pass filtering the raw image to suppress
noise before anything else happens. The choice of smoothing sigma is a direct trade-off: too small a
sigma leaves too much noise (risking false positives — noise spikes mistaken for particles); too
large a sigma merges or weakens real particles together (risking false negatives). A commonly used
practical rule of thumb sets sigma to roughly a third of the microscope's **Rayleigh resolution
limit** — the minimum separation at which two point sources can still be distinguished as separate —
though in practice the right sigma is often tuned empirically, as you will do below.

In [ ]:
# ── 2.1 compare several low-pass sigmas and their effect on apparent particle count ──
for s in [0.5, 2.0, 5.0]:
    filtered = blur_image(particles_image, gaussian_psf(s))
    print(f'sigma={s:>4}: filtered image max/mean ratio = {filtered.max() / filtered.mean():.2f}  '
          f'(higher = particles stand out more clearly from background)')

chosen_sigma = 1.3
lpf_image = blur_image(particles_image, gaussian_psf(chosen_sigma))
show_image_grid([particles_image, lpf_image], ['raw', f'low-pass filtered, sigma={chosen_sigma}'], ncols=2, figsize_scale=4)
plt.show()

✏️ **Your turn.** Try `chosen_sigma = 0.3` and `chosen_sigma = 6.0`. For each, describe in a
sentence what goes wrong for particle detection, connecting your answer to the "small sigma: more
false positives, large sigma: more false negatives" trade-off described above.

---
## 3 · Steps 2–3: local maxima, local background, and correspondence via Delaunay triangulation

In [ ]:
# ── 3.1 local maximum detection (candidate particles) and local minimum detection (candidate background) ──
from scipy.ndimage import maximum_filter, minimum_filter

mask_size = 5
local_max_mask = (lpf_image == maximum_filter(lpf_image, size=mask_size)) & (lpf_image > lpf_image.mean())
local_min_mask = (lpf_image == minimum_filter(lpf_image, size=mask_size))

maxima_coords = np.column_stack(np.nonzero(local_max_mask))
minima_coords = np.column_stack(np.nonzero(local_min_mask))
print('candidate local maxima (candidate particles):', len(maxima_coords))
print('candidate local minima (candidate background points):', len(minima_coords))

fig, ax = plt.subplots(figsize=(5, 5))
show_image(ax, lpf_image, 'candidate maxima (red) and minima (blue)')
ax.scatter(maxima_coords[:, 1], maxima_coords[:, 0], s=12, color='red', label='local maxima')
ax.scatter(minima_coords[:, 1], minima_coords[:, 0], s=4, color='cyan', alpha=0.5, label='local minima')
ax.legend(fontsize=7, loc='upper right')
plt.tight_layout(); plt.show()

🔍 **What just happened?** A $5\times5$ sliding-window maximum filter marks every pixel that is
the brightest in its own local neighborhood — candidate particles. A matching minimum filter marks
candidate local background points. There are far more candidate maxima than true particles (35):
noise and texture produce many spurious local maxima too, which is exactly why the next two steps
(correspondence and statistical selection) are necessary.

In [ ]:
# ── 3.2 establish correspondence between each maximum and its nearby minima via Delaunay triangulation ──
all_points = np.vstack([maxima_coords, minima_coords])
is_max = np.concatenate([np.ones(len(maxima_coords), dtype=bool), np.zeros(len(minima_coords), dtype=bool)])

tri = Delaunay(all_points)
# For each maximum, find its Delaunay-neighboring minima (points connected to it by a triangulation edge)
neighbors = {i: set() for i in range(len(all_points))}
for simplex in tri.simplices:
    for a in simplex:
        for b in simplex:
            if a != b:
                neighbors[a].add(b)

def local_background_for(max_idx):
    neighbor_idxs = [n for n in neighbors[max_idx] if not is_max[n]]
    if not neighbor_idxs:
        return lpf_image.mean()
    return lpf_image[all_points[neighbor_idxs][:, 0], all_points[neighbor_idxs][:, 1]].mean()

max_indices = np.where(is_max)[0]
backgrounds = np.array([local_background_for(i) for i in max_indices])
intensities = lpf_image[all_points[max_indices][:, 0], all_points[max_indices][:, 1]]
delta_I = intensities - backgrounds
print('for the first 5 candidate maxima: intensity - background =', np.round(delta_I[:5], 4))

🔍 **What just happened?** The Delaunay triangulation of all candidate maxima and minima
connects each point to its natural geometric neighbors — no arbitrary distance threshold needed. For
each candidate maximum, we then look up its Delaunay-neighboring minima and average their intensity
to estimate a *local*, spatially appropriate background (rather than a single global background
value, which would be a poor estimate in an image with uneven illumination).

### 📘 Lecture notes — Delaunay triangulation for correspondence

**Definition.** For a set of points in the plane, the Delaunay triangulation is the unique
triangulation in which no point lies inside the circumcircle of any triangle; among its useful
properties, it tends to favor large internal angles and connects points to their natural "nearest
neighbors" without requiring an arbitrary distance cutoff.

**Worked example.** Given scattered local maxima and local minima across a noisy fluorescence image,
Delaunay triangulation efficiently identifies, for each maximum, which minima are its true spatial
neighbors — exactly the correspondence needed to compute a spatially local background estimate.

**One memorable point.** *Delaunay triangulation gives you "nearest neighbor structure" for free,
without choosing a distance threshold by hand — a recurring advantage whenever point density varies
across an image.*

---
## 4 · Step 4: statistical feature selection with a t-test

Not every candidate maximum with $\Delta I = I_{\max} - I_{BG} > 0$ is a real particle — some of that
difference is just noise. The classical approach treats this as a **two-sample t-test**: are the
intensities "at the maximum" and "at the local background" distinguishable, given their variability?
Using $N$ local background samples with standard deviation $\sigma_{\Delta I}$, keep a candidate only
if $\Delta I \geq Q \cdot \sigma_{\Delta I}$ for a chosen selection quantile $Q$ — larger $Q$ means
stricter selection (fewer, higher-confidence particles).

In [ ]:
# ── 4.1 statistical selection: keep only maxima whose intensity is significantly above local background ──
sigma_delta_I = delta_I.std()
for Q in [1.0, 2.5, 4.0]:
    kept = delta_I >= Q * sigma_delta_I
    print(f'Q={Q:>4}: {kept.sum():>3} of {len(delta_I)} candidates kept')

Q_chosen = 2.5
kept_mask = delta_I >= Q_chosen * sigma_delta_I
kept_coords = all_points[max_indices][kept_mask]

fig, ax = plt.subplots(figsize=(5, 5))
show_image(ax, particles_image, f'statistically selected particles (Q={Q_chosen})')
ax.scatter(kept_coords[:, 1], kept_coords[:, 0], s=25, facecolors='none', edgecolors='lime', linewidths=1.5)
ax.scatter(true_positions[:, 1], true_positions[:, 0], s=8, color='red', marker='x', label='ground truth')
ax.legend(fontsize=7)
plt.tight_layout(); plt.show()
print('kept', kept_mask.sum(), 'candidates; true particle count was', len(true_positions))

🧪 **Ground-truth check.** Because the true particle positions are known (this is simulated
data), we can overlay them directly (red ✕ marks) against our selected detections (green circles)
and visually — or numerically, by matching each detection to its nearest ground-truth point — assess
how well the pipeline performed. Try computing, for each detection, its distance to the nearest
`true_positions` entry, and report how many detections are within, say, 3 pixels of a true particle.

✏️ **Your turn.** Sweep `Q_chosen` from 1.0 to 5.0 in a loop, and for each value record how many
candidates are kept. At what point does the kept count stabilize close to the true 35 particles?
This is exactly the kind of parameter-tuning judgment call a real analysis pipeline requires.

---
## 5 · Sub-pixel localization by Gaussian fitting

A detected particle's pixel-grid location (e.g. "row 42, column 87") is limited by the camera's
pixel size — if a particle's true center falls between pixels, the pixel-level detection cannot
distinguish that from a particle that happens to sit exactly on a pixel center. **Sub-pixel
localization** recovers a finer estimate by fitting a model (here, a Gaussian, approximating the
microscope's point-spread function) to the intensity pattern in a small window around each detected
maximum, and reporting the fitted center rather than the integer pixel coordinate.

In [ ]:
# ── 5.1 sub-pixel localization of every kept particle, via fit_gaussian_spot ──
window = 5
subpixel_positions = []
for cy, cx in kept_coords:
    cy, cx = int(cy), int(cx)
    y0, y1 = max(0, cy - window), min(particles_image.shape[0], cy + window + 1)
    x0, x1 = max(0, cx - window), min(particles_image.shape[1], cx + window + 1)
    patch = particles_image[y0:y1, x0:x1]
    local_cy, local_cx = fit_gaussian_spot(patch)
    subpixel_positions.append((y0 + local_cy, x0 + local_cx))
subpixel_positions = np.array(subpixel_positions)

pixel_level_error = np.abs(kept_coords - np.round(kept_coords)).mean()
subpixel_shift = np.abs(subpixel_positions - kept_coords).mean()
print(f'average |fractional part| of raw pixel coordinates (always 0 by construction): {pixel_level_error:.3f}')
print(f'average shift introduced by sub-pixel fitting: {subpixel_shift:.3f} pixels')

fig, ax = plt.subplots(figsize=(5, 5))
show_image(ax, particles_image, 'pixel-level (red) vs. sub-pixel (lime) detections')
ax.scatter(kept_coords[:, 1], kept_coords[:, 0], s=40, facecolors='none', edgecolors='red', label='pixel-level')
ax.scatter(subpixel_positions[:, 1], subpixel_positions[:, 0], s=15, color='lime', label='sub-pixel')
ax.legend(fontsize=7)
plt.tight_layout(); plt.show()

🔍 **What just happened?** `fit_gaussian_spot` uses intensity-weighted moments within a small
window to estimate a continuous (non-integer) center — a fast, closed-form approximation to full
Gaussian-fitting by optimization. The shift between the pixel-level and sub-pixel positions is
typically a fraction of a pixel, but that fraction matters enormously when, in Module 14, you need
to measure a transport *speed* from the frame-to-frame change in position: a systematic sub-pixel
bias, multiplied across many frames, can meaningfully distort a measured velocity.

🧪 **Ground-truth check.** Compute the distance from each sub-pixel position to its nearest true
position in `true_positions`, and compare the mean of these distances to the mean distance using the
pixel-level (un-refined) coordinates instead. Sub-pixel fitting should, on average, bring detections
closer to the true simulated positions.

### 📘 Lecture notes — sub-pixel localization

**Definition.** Sub-pixel localization estimates a particle's position with finer-than-one-pixel
precision by fitting a model of the expected intensity profile (often a Gaussian, approximating the
microscope's PSF) to the observed pixel intensities around a detected maximum, rather than reporting
only the integer pixel coordinate of the brightest pixel.

**Worked example.** Two historically important strategies for fitting: **exhaustive search** (very
finely oversample the candidate positions and intensity kernel, guaranteeing a good — though possibly
not globally optimal — solution, at high computational cost) and **optimization search** (use
gradient-based or related numerical optimization to find the best fit quickly, at the risk of getting
stuck in a local optimum); `fit_gaussian_spot`'s intensity-weighted-centroid approach is a fast,
closed-form compromise well suited to roughly symmetric, well-separated spots.

**One memorable point.** *The camera's pixel grid is a hard discretization of a continuous physical
world — sub-pixel methods exist precisely to partially recover the continuous information that
discretization otherwise throws away.*

## 📌 Key takeaways

- A "particle" in bioimaging is operationally a local intensity maximum significantly above its local background — this simple definition underlies a complete, multi-step detection pipeline.
- The classical pipeline is: low-pass filter → local maxima/minima → establish correspondence (e.g. via Delaunay triangulation) → statistically select true features (e.g. via a t-test-style threshold) → refine to sub-pixel precision.
- Delaunay triangulation gives a natural, threshold-free way to find each maximum's nearby background points.
- Sub-pixel localization recovers precision beyond the camera's pixel grid, which matters enormously for any downstream measurement (like tracking speed) that depends on small, repeated position differences.

## ✅ Checkpoint

**1.** Why is a single global background value usually a worse choice than a spatially local
background estimate?

<details><summary>Answer</summary>
Because illumination and background fluorescence are rarely perfectly uniform across a real image;
a local estimate (e.g. from Delaunay-neighboring minima) adapts to spatially varying background,
while a single global value does not.
</details>

**2.** What does increasing the selection quantile $Q$ do to the number of detected particles, and
why?

<details><summary>Answer</summary>
It decreases the number of detections kept, because a higher $Q$ requires a candidate's intensity to
exceed its local background by a larger multiple of the background noise's standard deviation —
a stricter, higher-confidence threshold.
</details>

**3.** Why can't pixel-level detection alone give you a particle's true position, even in a
noise-free image?

<details><summary>Answer</summary>
Because the camera's finite pixel size discretizes position; a particle whose true center falls
between pixel centers is still reported only at the resolution of the pixel grid unless a sub-pixel
method is used.
</details>

## 🧠 Questions & discussion

- The t-test-based selection step assumes the local background intensities are roughly normally
  distributed. What might go wrong with this assumption near a very bright, structured region of an
  image (for example, near another nearby particle)?
- Gaussian fitting by intensity-weighted moments is fast but can be biased when two particles are
  close enough that their PSFs overlap. How might you detect, from the data itself, when this bias
  is likely to be a problem?
- Why might exhaustive search (oversampling) be preferred over optimization search in some
  situations, despite being slower, as the lecture notes mention?

## 🚀 Bonus — going further

- Implement the "optimization search" alternative to `fit_gaussian_spot`, using
  `scipy.optimize.minimize` to directly fit a 2-D Gaussian's center, amplitude, and sigma to a patch,
  and compare its accuracy (against ground truth) to the intensity-weighted-moment approach.
- Re-run the whole pipeline with `association=1.0` in `simulate_two_channel_image` and detect
  particles in channel B as well; this sets you up directly for the colocalization analysis in
  Module 18.
- Investigate how detection performance (fraction of true particles recovered, within some distance
  tolerance) changes as you vary the simulated noise level — build a small sweep and plot the
  result.

## 📝 Lab note

```text
Module: 05 — Keypoints and Particles
Date:
My chosen (sigma, Q) pair and why I settled on it:
A failure case I noticed (missed or spurious detection) and a hypothesis for why:
```

## 📚 References

1. Sbalzarini, I.F., Koumoutsakos, P. (2005). Feature point tracking and trajectory analysis for
   video imaging in cell biology. *Journal of Structural Biology*, 151(2), 182–195.
2. Cheezum, M.K., Walker, W.F., Guilford, W.H. (2001). Quantitative comparison of algorithms for
   tracking single fluorescent particles. *Biophysical Journal*, 81(4), 2378–2388.
3. Delaunay, B. (1934). Sur la sphère vide. *Bulletin de l'Académie des Sciences de l'URSS*, 7,
   793–800.
4. Thomann, D. et al. (2002). Automatic fluorescent tag localization II: improvement in
   super-resolution by relative tracking. *Journal of Microscopy*, 211(3), 230–248.
5. Welch, B.L. (1947). The generalization of 'Student's' problem when several different population
   variances are involved. *Biometrika*, 34(1/2), 28–35.

## ➡️ Next

Module 06 covers a complementary kind of structure — edges, lines, and curves — using the Canny
pipeline, the Hough transform, and Steger's algorithm for detecting curvilinear structures like
filaments and microtubules.